# Example: Threads Simple Use Case (Array Sum)
We use two thread to sum the array of 10 numbers.

In [3]:
%%bash
cat <<EOF > code.c

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

#define SIZE 10

int numbers[SIZE] = {1,2,3,4,5,6,7,8,9,10};

// Thread start routine signature 
// void* thread_func(void* arg);

void* sum_half(void* arg) {
    int thread_id = *(int*)arg;
    
    // (int*)arg casts arg to int
    // *(int*)arg dereferences arg
    
    int start = thread_id * (SIZE/2);
    int end   = start + (SIZE/2);
    int sum = 0;

    for(int i = start; i < end; i++) {
        sum += numbers[i];
    }

    int* result = malloc(sizeof(int));
    *result = sum; // The value of sum is now saved inside the heap memory
    return result; // returns the pointer to that heap memory   
                   // int*  →  void* No cast needed here
}

int main() {
    pthread_t threads[2];
    int ids[2] = {0, 1};
    int total = 0;

    for(int i = 0; i < 2; i++) {
        pthread_create(&threads[i], NULL, sum_half, &ids[i]);
    }

    for(int i = 0; i < 2; i++) {
        int* partial;
        pthread_join(threads[i], (void**)&partial); 
        
        // signature:pthread_join(pthread_t thread, void **retval);
        
        total += *partial;
        free(partial); // free allocated memory
    }

    printf("Total sum = %d\n", total);
    return 0;
}

EOF

gcc code.c -o code
./code

Total sum = 55


# Deadlock:
A deadlock in C is when two or more threads are stuck forever because each is waiting for a resource locked by the other.

# Example
- Student 1: "I locked Room A. Now I need Room B too."
- Student 2: "I locked Room B. Now I need Room A too."

They both wait... FOREVER. Neither finishes. 

In [6]:
%%bash
cat <<EOF > code.c

#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

pthread_mutex_t roomA, roomB;  // Two rooms, two locks

void* student1(void* arg) {
    pthread_mutex_lock(&roomA);
    printf("Student 1: Locked Room A\n");
    sleep(1);  // Using Room A
    
    pthread_mutex_lock(&roomB);  // Needs Room B → WAITS FOREVER
    printf("Student 1: Locked Room B\n");  // NEVER prints
    
    pthread_mutex_unlock(&roomB);
    pthread_mutex_unlock(&roomA);
    return NULL;
}

void* student2(void* arg) {
    pthread_mutex_lock(&roomB);
    printf("Student 2: Locked Room B\n");
    sleep(1);  // Using Room B
    
    pthread_mutex_lock(&roomA);  // Needs Room A → WAITS FOREVER
    printf("Student 2: Locked Room A\n");  // NEVER prints
    
    pthread_mutex_unlock(&roomA);
    pthread_mutex_unlock(&roomB);
    return NULL;
}

int main() {
    pthread_mutex_init(&roomA, NULL);
    pthread_mutex_init(&roomB, NULL);
    
    pthread_t s1, s2;
    pthread_create(&s1, NULL, student1, NULL);
    pthread_create(&s2, NULL, student2, NULL);
    
    pthread_join(s1, NULL);  // NEVER finishes
    pthread_join(s2, NULL);  // NEVER finishes
    
    printf("Both students finished!\n");  // NEVER prints
    return 0;
}

EOF

gcc code.c -o code
./code

Process is interrupted.


# trylock:
In C with pthreads, pthread_mutex_lock will block until the mutex becomes available, while pthread_mutex_trylock is non-blocking—it immediately returns if the mutex is already locked. This makes trylock useful when you want to avoid waiting and instead handle the situation differently if the lock is busy.

In [6]:
%%bash
cat <<EOF > code.c

#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

pthread_mutex_t roomA, roomB;

void* student1(void* arg) {
    pthread_mutex_lock(&roomA);
    printf("Student 1: Locked Room A - Studying Math\n");
    sleep(3);  // Long study session
    
    printf("Student 1: Need Room B for reference book...\n");
    pthread_mutex_lock(&roomB);  // WAITS FOREVER - DEADLOCK!
    printf("Student 1: Got Room B! Complete!\n");
    
    pthread_mutex_unlock(&roomB);
    pthread_mutex_unlock(&roomA);
    return NULL;
}

void* student2(void* arg) {
    pthread_mutex_lock(&roomB);
    printf("Student 2: Locked Room B - Drawing Art\n");
    sleep(1);
    
    // TRYLOCK - The smart student!
    printf("Student 2: Need Room A for art supplies...\n");
    
    if (pthread_mutex_trylock(&roomA) == 0) {
        printf("Student 2: Got Room A! Complete!\n");
        pthread_mutex_unlock(&roomA);
    } else {
        printf("Student 2: Room A is busy! Can't wait.\n");
        printf("Student 2: Instead, I'll go to the library to read.\n");
        pthread_mutex_unlock(&roomB);
        printf("Student 2: Released Room B. Will try again after library.\n");
        
        // DO SOMETHING ELSE INSTEAD OF WAITING!
        printf("Student 2: Reading at library for 2 seconds...\n");
        sleep(2);
        
        // Try again later
        printf("Student 2: Trying again...\n");
        pthread_mutex_lock(&roomB);
        if (pthread_mutex_trylock(&roomA) == 0) {
            printf("Student 2: Got both rooms this time! Complete!\n");
            pthread_mutex_unlock(&roomA);
            pthread_mutex_unlock(&roomB);
        }
    }
    return NULL;
}

int main() {
    pthread_mutex_init(&roomA, NULL);
    pthread_mutex_init(&roomB, NULL);
    
    pthread_t s1, s2;
    pthread_create(&s1, NULL, student1, NULL);
    pthread_create(&s2, NULL, student2, NULL);
    
    pthread_join(s1, NULL);
    pthread_join(s2, NULL);
    
    printf("\n Both students finished!\n");
    return 0;
}

EOF

gcc code.c -o code
./code

Student 2: Locked Room B - Drawing Art
Student 1: Locked Room A - Studying Math
Student 2: Need Room A for art supplies...
Student 2: Room A is busy! Can't wait.
Student 2: Instead, I'll go to the library to read.
Student 2: Released Room B. Will try again after library.
Student 2: Reading at library for 2 seconds...
Student 1: Need Room B for reference book...
Student 1: Got Room B! Complete!
Student 2: Trying again...
Student 2: Got both rooms this time! Complete!

 Both students finished!


# Points to remember:
- trylock alone doesn’t solve deadlock universally — it just gives you a chance to design smarter behavior.

- In real systems, you’d often combine trylock with timeouts, lock ordering, or backoff strategies.

- Busy-waiting with repeated trylock calls can waste CPU cycles if not managed carefully.

# Semaphores
A semaphore is a synchronization variable used to control access to a shared resource. 

It works like a counter that represents the number of available “permits” (or “tickets”). Threads must acquire a permit before entering and release it when done.

 **The Two Fundamental Operations:**
- sem_wait()	Decrease counter. If counter < 0, block/sleep	
- sem_post()	Increase counter. If threads are waiting, wake one


**Semaphore types:**

- Binary semaphore	0 or 1	Like a lock (mutex)
- Counting semaphore	0 to N	Manage multiple identical resources

In [7]:
%%bash
cat <<EOF > code.c

#include <stdio.h>
#include <pthread.h>
#include <semaphore.h>
#include <unistd.h>

sem_t sem;  // Semaphore to control access

void* print_hello(void* arg) {
    int thread_id = *(int*)arg;
    
    sem_wait(&sem);  // Wait for permission (LOCK)
    
    // CRITICAL SECTION - Only ONE thread can be here at a time
    printf("Hello from thread %d\n", thread_id);
    sleep(3);  // Simulate some work
    
    sem_post(&sem);  // Release permission (UNLOCK)
    
    return NULL;
}

int main() {
    pthread_t t1, t2;
    int id1 = 1, id2 = 2;
    
    // Initialize semaphore with value 1 (binary semaphore = lock)
    sem_init(&sem, 0, 1);
    
    // Create two threads
    pthread_create(&t1, NULL, print_hello, &id1);
    pthread_create(&t2, NULL, print_hello, &id2);
    
    // Wait for both threads to finish
    pthread_join(t1, NULL);
    pthread_join(t2, NULL);
    
    // Clean up
    sem_destroy(&sem);
    
    return 0;
}



EOF

gcc code.c -o code
./code

Hello from thread 1
Hello from thread 2


In [5]:
%%bash
cat <<EOF > code.c

#include <stdio.h>
#include <pthread.h>
#include <semaphore.h>
#include <unistd.h>

sem_t sem;  // Counting semaphore (allows 2 threads at once)

void* print_hello(void* arg) {
    int thread_id = *(int*)arg;
    
    printf("Thread %d: Trying to enter...\n", thread_id);
    sem_wait(&sem);  // Request permission (decrease counter)
    
    // CRITICAL SECTION - Only 2 threads can be here at a time
    printf(">>> Thread %d: INSIDE CRITICAL SECTION! <<<\n", thread_id);
    printf("Hello from thread %d\n", thread_id);
    sleep(2);  // Simulate work
    
    sem_post(&sem);  // Release permission (increase counter)
    printf("Thread %d: LEFT critical section\n", thread_id);
    
    return NULL;
}

int main() {
    pthread_t t1, t2, t3, t4;
    int id1 = 1, id2 = 2, id3 = 3, id4 = 4;
    
    // Initialize semaphore with value 2 (allows 2 threads at once)
    sem_init(&sem, 0, 2);
    
    // Create FOUR threads
    pthread_create(&t1, NULL, print_hello, &id1);
    pthread_create(&t2, NULL, print_hello, &id2);
    pthread_create(&t3, NULL, print_hello, &id3);
    pthread_create(&t4, NULL, print_hello, &id4);
    
    // Wait for all threads to finish
    pthread_join(t1, NULL);
    pthread_join(t2, NULL);
    pthread_join(t3, NULL);
    pthread_join(t4, NULL);
    
    // Clean up
    sem_destroy(&sem);
    
    return 0;
}

EOF

gcc code.c -lpthread -o code
./code

Thread 1: Trying to enter...
>>> Thread 1: INSIDE CRITICAL SECTION! <<<
Hello from thread 1
Thread 2: Trying to enter...
>>> Thread 2: INSIDE CRITICAL SECTION! <<<
Hello from thread 2
Thread 3: Trying to enter...
Thread 4: Trying to enter...
Thread 1: LEFT critical section
Thread 2: LEFT critical section
>>> Thread 3: INSIDE CRITICAL SECTION! <<<
Hello from thread 3
>>> Thread 4: INSIDE CRITICAL SECTION! <<<
Hello from thread 4
Thread 3: LEFT critical section
Thread 4: LEFT critical section
